# AgriMate — 番茄病虫害 4 模型对比实验
Colab T4 GPU · PlantVillage 直连 · 番茄 10 类

In [ ]:
!pip install transformers datasets torchvision accelerate -q

In [ ]:
import torch, json, os
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification, CLIPProcessor, CLIPModel
from datasets import load_dataset
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB')

## 加载 PlantVillage — 只取番茄 10 类

In [ ]:
TOMATO_CLASSES = ['细菌性斑点病','早疫病','晚疫病','叶霉病','斑枯病',
                   '红蜘蛛','靶斑病','黄化曲叶病毒','花叶病毒','健康']

print('Loading PlantVillage...')
full_ds = load_dataset('LamTNguyen/PlantVillage', split='train')
print(f'Total: {len(full_ds)} images')

# dataset label 5-14 = tomato
tomato_idx = [i for i in range(len(full_ds)) if full_ds[i]['label'] in range(5, 15)]
print(f'Tomato: {len(tomato_idx)} images')

train_n = int(0.8 * len(tomato_idx))
test_n = len(tomato_idx) - train_n
train_idx, test_idx = random_split(tomato_idx, [train_n, test_n],
                                   generator=torch.Generator().manual_seed(42))
print(f'Train: {len(train_idx)} | Test: {len(test_idx)}')

## Dataset & DataLoader

In [ ]:
class TomatoDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = list(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        sample = self.dataset[self.indices[i]]
        img = sample['image'].convert('RGB')
        label = sample['label'] - 5  # -> 0..9
        if self.transform:
            img = self.transform(img)
        return img, label

# ImageNet 标准预处理
train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
test_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

train_ds = TomatoDataset(full_ds, train_idx, train_tf)
test_ds = TomatoDataset(full_ds, test_idx, test_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## 评测函数

In [ ]:
def evaluate(model, dataloader, desc='Eval'):
    model.eval()
    correct = torch.zeros(10)
    total = torch.zeros(10)
    with torch.no_grad():
        for imgs, labels in tqdm(dataloader, desc=desc):
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            logits = out.logits if hasattr(out, 'logits') else out
            preds = logits.argmax(dim=1)
            for c in range(10):
                mask = (labels == c)
                correct[c] += (preds[mask] == labels[mask]).sum()
                total[c] += mask.sum()
    per_class = {TOMATO_CLASSES[i]: round((correct[i]/total[i]*100).item(), 1)
                 if total[i]>0 else 0 for i in range(10)}
    overall = round((correct.sum()/total.sum()*100).item(), 2)
    return overall, per_class

## 模型 1: ResNet50 (已微调)

In [ ]:
print('Loading ResNet50...')
r50 = AutoModelForImageClassification.from_pretrained(
    'SanketJadhav/PlantDiseaseClassifier-Resnet50'
).to(device)
# 模型输出 38 类，取番茄部分 (id 28~37)
r50.orig = r50.forward
r50.forward = lambda x: r50.orig(x)[:, 28:38]

r50_acc, r50_pc = evaluate(r50, test_loader, 'ResNet50')
print(f'ResNet50 Accuracy: {r50_acc}%')

## 模型 2: MobileNetV2 (已微调)

In [ ]:
from huggingface_hub import snapshot_download

print('Loading MobileNetV2...')
mdir = snapshot_download(
    'linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification',
    local_dir='./mbv2', local_dir_use_symlinks=False
)
# 修复过时的 preprocessor config
pp = os.path.join(mdir, 'preprocessor_config.json')
with open(pp) as f:
    cfg = json.load(f)
cfg.pop('image_processor_type', None)
cfg.pop('feature_extractor_type', None)
with open(pp, 'w') as f:
    json.dump(cfg, f, indent=2)

mbv2 = AutoModelForImageClassification.from_pretrained(mdir).to(device)
mbv2.orig = mbv2.forward
mbv2.forward = lambda x: mbv2.orig(x)[:, 28:38]

mb_acc, mb_pc = evaluate(mbv2, test_loader, 'MobileNetV2')
print(f'MobileNetV2 Accuracy: {mb_acc}%')

## 模型 3: Swin Tiny — 微调 10 Epochs

In [ ]:
print('Loading Swin Tiny...')
swin = AutoModelForImageClassification.from_pretrained(
    'microsoft/swin-tiny-patch4-window7-224',
    num_labels=10, ignore_mismatched_sizes=True
).to(device)

opt = torch.optim.AdamW(swin.parameters(), lr=1e-4)
crit = nn.CrossEntropyLoss()

for ep in range(10):
    swin.train()
    loss_sum = 0
    for imgs, labels in tqdm(train_loader, desc=f'Swin Epoch {ep+1}/10'):
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        loss = crit(swin(imgs).logits, labels)
        loss.backward()
        opt.step()
        loss_sum += loss.item()
    print(f'  Loss: {loss_sum/len(train_loader):.4f}')

swin_acc, swin_pc = evaluate(swin, test_loader, 'Swin Tiny')
print(f'Swin Tiny Accuracy (after fine-tune): {swin_acc}%')

## 模型 4: CLIP ViT-B/32 — 线性探测 5 Epochs

In [ ]:
print('Loading CLIP...')
clip_m = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
for p in clip_m.vision_model.parameters():
    p.requires_grad = False  # freeze vision encoder

class CLIPLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision = clip_m.vision_model
        self.fc = nn.Linear(768, 10)
    def forward(self, x):
        return self.fc(self.vision(x).pooler_output)

clip_clf = CLIPLinear().to(device)
opt_clip = torch.optim.AdamW(clip_clf.fc.parameters(), lr=1e-3)

# CLIP 专用预处理
clip_tf = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize((0.48145466,0.4578275,0.40821073),
                         (0.26862954,0.26130258,0.27577711))
])
clip_train_ds = TomatoDataset(full_ds, train_idx, clip_tf)
clip_test_ds = TomatoDataset(full_ds, test_idx, clip_tf)
clip_train_ld = DataLoader(clip_train_ds, batch_size=32, shuffle=True)
clip_test_ld = DataLoader(clip_test_ds, batch_size=32, shuffle=False)

for ep in range(5):
    clip_clf.train()
    ls = 0
    for imgs, labels in tqdm(clip_train_ld, desc=f'CLIP Epoch {ep+1}/5'):
        imgs, labels = imgs.to(device), labels.to(device)
        opt_clip.zero_grad()
        loss = crit(clip_clf(imgs), labels)
        loss.backward()
        opt_clip.step()
        ls += loss.item()
    print(f'  Loss: {ls/len(clip_train_ld):.4f}')

clip_acc, clip_pc = evaluate(clip_clf, clip_test_ld, 'CLIP')
print(f'CLIP Accuracy (linear probe): {clip_acc}%')

## 结果汇总

In [ ]:
print('='*60)
print('番茄病虫害 4 模型对比')
print('='*60)

results = [
    ('ResNet50 (已微调)', r50_acc, r50_pc),
    ('MobileNetV2 (已微调)', mb_acc, mb_pc),
    ('Swin Tiny (微调后)', swin_acc, swin_pc),
    ('CLIP ViT-B/32 (线性探测)', clip_acc, clip_pc),
]

# 总体准确率
print(f'\n{"Model":<30s} Accuracy')
print('-'*45)
for n, a, _ in results:
    print(f'  {n:<30s} {a:>5.1f}%')

# Per-class 对比
print(f'\n{"Class":<12s}', end='')
for n, _, _ in results:
    print(f'{" ":>5s}{n.split()[0]:>6s}', end='')
print()
for i, cn in enumerate(TOMATO_CLASSES):
    print(f'  {cn:<12s}', end='')
    for _, _, pc in results:
        print(f'    {pc[cn]:>5.1f}%', end='')
    print()

# 保存
with open('tomato_4model_results.json', 'w') as f:
    json.dump([{'name':n, 'accuracy':a, 'per_class':p} for n,a,p in results],
              f, indent=2, ensure_ascii=False)
print('\nResults saved to tomato_4model_results.json')
print('Done!')